# Rosalution Analysis 'Test' - TRMT61A

In [8]:
%load_ext autoreload
%autoreload 2

from core.api.rosalution import rosalution_authenticate, Rosalution


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
request_auth_token = rosalution_authenticate()

🔒 Requesting Auth from Rosalution at https://cgds.uab.edu/rosalution/api/auth/token.......


🔒 Authenticating with Rosalution...Complete


## Queries Rosalution Analysis data


In [10]:
rosalution_test_example = Rosalution("CPAM0002", request_auth_token)
test_omic_units = rosalution_test_example.get_genomic_units()

print(test_omic_units)
print(test_omic_units.get('genes'))
print(rosalution_test_example.get_analysis_hpo_terms())
print(rosalution_test_example.get_models_of_interest())

https://cgds.uab.edu/rosalution/api/analysis/CPAM0002/genomic_units
{
  "genes": {
    "VMA21": [
      "NM_001017980.3:c.164G>T(p.Gly55Val)"
    ]
  },
  "variants": [
    "NM_001017980.3:c.164G>T(p.Gly55Val)"
  ]
}
{'gene': ['VMA21'], 'hgvs_variant': ['NM_001017980.3:c.164G>T']}
None
['HP:0003198', 'HP:0003797', 'HP:0003325', 'HP:0008997', 'HP:0008994', 'HP:0001288', 'HP:0009060', 'HP:0004303', 'HP:0012103', 'HP:0003736']
['Zebrafish', 'Mouse']


## Queries Annotations for gene

In [11]:
from IPython.display import display, Markdown

from core.api.alliance_genome import AllianceGenome

gene_list = test_omic_units['gene']
annotations_cache = {}

for gene in gene_list:
    annotations_cache[gene] = rosalution_test_example.get_annotations('gene', gene)
    
    display(Markdown(f"### Gene HGNC ID from Rosalution"))

    gene_hgnc_id = rosalution_test_example.get_annotation('gene', gene, 'HGNC_ID')
    print(gene_hgnc_id)

    display(Markdown(f"### Querying Orthologous Animal Model Identifiers from Alliance Genome"))
    ortholog_ids = AllianceGenome.orthologs(gene_hgnc_id)
    print(ortholog_ids)
    display(Markdown("> **note:** orthologous gene id's searched with a stringent filter applied"))


### Gene HGNC ID from Rosalution

HGNC:22082


### Querying Orthologous Animal Model Identifiers from Alliance Genome

['WB:WBGene00011115', 'MGI:1914298', 'ZFIN:ZDB-GENE-081104-272', 'Xenbase:XB-GENE-5730849', 'RGD:1566155', 'FB:FBgn0036998', 'Xenbase:XB-GENE-6255895', 'FB:FBgn0259705', 'SGD:S000003337']


> **note:** orthologous gene id's searched with a stringent filter applied

In [12]:
from core.api.diopt import Diopt

MODEL_SYSTEM_NAMESPACE = [
    ("Xenbase", "Xenopus", 8364),
    ("WB", "C. elegans", 6239),
    ("MGI", "Mouse", 1914298),
    ("ZFIN", "Zebrafish", 7955),
    ("RGD", "Rat", 10116)
]
for gene in gene_list:
    rosalution_test_example.get_annotations('gene', gene)
    gene_hgnc_id = rosalution_test_example.get_annotation('gene', gene, 'HGNC_ID')
    ortholog_ids = AllianceGenome.orthologs(gene_hgnc_id)

    gene_has_models = {}
    for model_namespace, animal_model, _ in MODEL_SYSTEM_NAMESPACE:
        models_by_id = [model_id for model_id in ortholog_ids if model_namespace in model_id]
        for model_system_id in models_by_id:
            has_models = AllianceGenome.has_models(model_system_id)
            if has_models:
                gene_has_models.add(animal_model) if gene in gene_has_models else gene_has_models.setdefault(gene, {animal_model})
    
    display(Markdown(f"### Following Rosalution Analysis Genes Have Animal Models in Alliance Genome"))
    print(gene_has_models)
    
    display(Markdown(f"### The degree of conservation of '{gene}'gene across animal models"))
    entrez_gene_id = rosalution_test_example.get_annotation('gene', gene, 'Entrez Gene Id')
    entrez_gene_id_string = str(entrez_gene_id)
    diopt_api_query = Diopt.best_score(entrez_gene_id)
    
    diopt_results_by_gene = diopt_api_query['results'][entrez_gene_id_string]
    diopt_score_by_species = {}
    for model_space, animal_model, species_id in MODEL_SYSTEM_NAMESPACE:
        diopt_score_by_species[animal_model] =  [ diopt_result for diopt_result in diopt_results_by_gene.values() if diopt_result["species_id"] == species_id ]
        for scores_by_species in diopt_score_by_species[animal_model]:
            print(
                f"Degree of {animal_model}'s gene  {scores_by_species['symbol']} ({scores_by_species['species_specific_geneid']}) "
                f"with score {scores_by_species['score']}/{scores_by_species['max_score']} "
                f"{scores_by_species['confidence']} confidence"
            )

display(Markdown("> **note:** queried Diopt API for its 'best scores'"))

### Following Rosalution Analysis Genes Have Animal Models in Alliance Genome

{'VMA21': {'Zebrafish'}}


### The degree of conservation of 'VMA21'gene across animal models

Degree of Xenopus's gene  vma21 (XB-GENE-5730849) with score 13/14 high confidence
Degree of C. elegans's gene  pdcd-2 (WBGene00011116) with score 9/13 high confidence
Degree of C. elegans's gene  R07E5.19 (WBGene00303105) with score 8/13 moderate confidence
Degree of C. elegans's gene  R07E5.7 (WBGene00011115) with score 3/13 moderate confidence
Degree of Zebrafish's gene  vma21 (ZDB-GENE-081104-272) with score 14/15 high confidence
Degree of Rat's gene  Vma21 (1566155) with score 15/15 high confidence


> **note:** queried Diopt API for its 'best scores'

## For Rosalution Test, queries Monarch's Neo4j Graph Database with Rosalution 'Test' hpo terms

In [ ]:
from core.api.monarch import RosalutionMonarch
from core.visualization.phenotype import sankey_phenotype_system_modelsystem, sankey_homologous_phenotypes_and_modelsystem, sankey_homologous_phenotypes_and_modelsystem_close_only

test_descriptive_hpo_terms = rosalution_test_example.get_analysis_descriptive_hpo_terms()

with RosalutionMonarch(rosalution_test_example) as monarch_rosalutiontest:
    test_orthologous_phenotype_systems = monarch_rosalutiontest.rosalution_orthologous_phenotypic_systems()
    test_homologous_phenotypic_terms =  monarch_rosalutiontest.rosalution_homologous_phenotypic_terms()

    print(test_descriptive_hpo_terms)
    print(test_orthologous_phenotype_systems)
    print(test_homologous_phenotypic_terms)

    # test_hpo_terms_within_systems = sankey_phenotype_system_modelsystem(test_orthologous_phenotype_systems, "Test")
    # test_hpo_terms_within_systems.show()
    # test_hpo_terms_within_systems.write_html("../plots/test-phenotype-to-system-terms.html")

    sankey_example_plot = sankey_homologous_phenotypes_and_modelsystem(test_homologous_phenotypic_terms, "Test")
    sankey_example_plot.show()

    closest_example_plot = sankey_homologous_phenotypes_and_modelsystem_close_only(test_homologous_phenotypic_terms, "Test")
    closest_example_plot.show()
    # closest_example_plot.write_html("./plots/test-phenotype-to-homologus-terms.html")

['HP:0003198', 'HP:0003797', 'HP:0003325', 'HP:0008997', 'HP:0008994', 'HP:0001288', 'HP:0009060', 'HP:0004303', 'HP:0012103', 'HP:0003736']
[{'terms': ['Gait disturbance'], 'system': 'nervous system phenotype', 'system_source': 'UPHENO', 'modelsystem': ['XPO', 'PLANP', 'MP', 'HP', 'ZP']}, {'terms': ['Myopathy', 'Limb-girdle muscle weakness', 'Autophagic vacuoles', 'Limb-girdle muscle atrophy', 'Abnormal muscle fiber morphology', 'Proximal lower limb muscle weakness', 'Proximal upper limb muscle weakness', 'Scapular muscle atrophy'], 'system': 'musculature phenotype', 'system_source': 'UPHENO', 'modelsystem': ['MP', 'HP', 'ZP']}, {'terms': ['Abnormality of the mitochondrion'], 'system': 'Abnormality of metabolism/homeostasis', 'system_source': 'HP', 'modelsystem': ['HP']}]
[{'terms': ['Gait disturbance'], 'distance': 3, 'closeHomologousTerm': 'abnormal nervous system physiology', 'homologousModelSystem': ['MP']}, {'terms': ['Gait disturbance'], 'distance': 3, 'closeHomologousTerm': 'ne

### Analysis' HPO Terms Mapped To Orthologous Genes and Tissues Expressed In

In [14]:
from core.api.monarch import MonarchKnowledgeGraph
from core.visualization.anatomy import sankey_gene_to_expressed_in_anatomy
import json

gene_list = test_omic_units['gene']
annotations_cache = {}
orthologous_gene_expression_in = {}

for gene in gene_list:
    annotations_cache[gene] = rosalution_test_example.get_annotations('gene', gene)
    gene_hgnc_id = rosalution_test_example.get_annotation('gene', gene, 'HGNC_ID')
    orthologous_gene_expression_in[gene] = MonarchKnowledgeGraph.orthologous_gene_expression_of_tissue(gene_hgnc_id)
    sankey_plot = sankey_gene_to_expressed_in_anatomy( orthologous_gene_expression_in[gene])
    sankey_plot.show()

building source and targets


In [15]:
annotation_manifest = rosalution_test_example.get_annotation_manifest()
annotation_manifest

[{'unit': 'VMA21',
  'manifest': [{'Frog_General_Xenbase_Database_url': {'data_source': 'Alliance Genome',
     'version': '8.1.0'}},
   {'HGNC_GENE_NAMES_CALL_CACHE': {'data_source': 'HGNC_GeneNames',
     'version': '2025-05-23'}},
   {'ANIMAL_MODEL_IDS_VIA_AUTOCOMPLETE_ALLIANCE_GENOME_CACHE': {'data_source': 'Alliance Genome',
     'version': '8.1.0'}},
   {'Ensembl Gene Id': {'data_source': 'HGNC_GeneNames',
     'version': '2025-05-23'}},
   {'HGNC_ID': {'data_source': 'HGNC_GeneNames', 'version': '2025-05-23'}},
   {'Rat Gene Identifier': {'data_source': 'Alliance Genome',
     'version': '8.1.0'}},
   {'Mouse Gene Identifier': {'data_source': 'Alliance Genome',
     'version': '8.1.0'}},
   {'Zebrafish Gene Identifier': {'data_source': 'Alliance Genome',
     'version': '8.1.0'}},
   {'Entrez Gene Id': {'data_source': 'HGNC_GeneNames',
     'version': '2025-05-23'}},
   {'Gene Summary': {'data_source': 'Alliance Genome', 'version': '8.1.0'}},
   {'HPO_NCBI_GENE_ID': {'data_sourc

In [16]:
#Previous attempt to try and create an HTML doc from the notebook's output
#!jupyter nbconvert --to html single_rosalution_analysis.ipynb